## Import

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"   # single-threaded so parallel jobs do not fight over cores

import json, os, time, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             log_loss, average_precision_score, mean_squared_error, mean_absolute_error, r2_score)
warnings.filterwarnings("ignore")
torch.set_num_threads(1)

## Setup

In [2]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
K, REPEATS = 5, 3
os.makedirs("weights_V5_1", exist_ok=True)

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()  # 2213 rows (target must not be imputed)
print("cleaned:", df.shape, "| Income max:", f"{df['Income'].max():,.0f}", "| median:", f"{df['Income'].median():,.0f}")
REG_FEATURES = ["NumCatalogPurchases","MntMeatProducts","MntWines","NumWebVisitsMonth","MntSweetProducts",
    "MntFruits","NumWebPurchases","MntGoldProds","NumStorePurchases","MntFishProducts","Kidhome","Age","NumDealsPurchases"]
X_full = df[REG_FEATURES].values; y_raw = df["Income"].values.astype(float)
print(f"features: {len(REG_FEATURES)} dims (all numeric)")

cleaned: (2212, 28) | Income max: 162,397 | median: 51,371
features: 13 dims (all numeric)


## Feature Interaction

In [4]:
# Feature interaction hand-off (Feature Interaction section of regression.ipynb):
# Teenhome interacts with most spend features but is not in the 13-feature model set.
EXTRA = ["Teenhome"]
COLS_ALL = REG_FEATURES + EXTRA
X_all = df[COLS_ALL].values.astype(float)

def top_pairs(csv_path, cols, k, must_include=None, exclude=None):
    """Top-k significant (q < 0.05) pairs by effect size where both features are in `cols`."""
    t = pd.read_csv(csv_path)
    t = t[(t.q < 0.05) & t.A.isin(cols) & t.B.isin(cols)]
    if must_include is not None:
        t = t[t.A.eq(must_include) | t.B.eq(must_include)]
    if exclude is not None:
        t = t[~(t.A.eq(exclude) | t.B.eq(exclude))]
    t = t.sort_values("effect", ascending=False)
    return list(zip(t.A.head(k), t.B.head(k)))

PAIRS_13 = top_pairs("results/interaction_reg.csv", REG_FEATURES, 4)
PAIRS_T = top_pairs("results/interaction_reg.csv", COLS_ALL, 5, must_include="Teenhome")
print("pairs (13 features):", PAIRS_13); print("pairs (with Teenhome):", PAIRS_T)
SETS = {
    "13 features":                    dict(cols=REG_FEATURES, pairs=[]),
    "+ Teenhome (14)":                dict(cols=COLS_ALL, pairs=[]),
    "13 features + 4 pair nets":      dict(cols=REG_FEATURES, pairs=PAIRS_13),
    "+ Teenhome + 5 Teenhome pairs":  dict(cols=COLS_ALL, pairs=PAIRS_T),
}

def prep_split(tr_idx, te_idx, set_name):
    cols = SETS[set_name]["cols"]
    ci = [COLS_ALL.index(c) for c in cols]
    sc = StandardScaler().fit(X_all[tr_idx][:, ci])
    return sc.transform(X_all[tr_idx][:, ci]), sc.transform(X_all[te_idx][:, ci]), y_raw[tr_idx], y_raw[te_idx]

idx_all = np.arange(len(y_raw))
tr_idx, te_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED)
for s_ in SETS:
    a_, b_, _, _ = prep_split(tr_idx, te_idx, s_)
    print(f"{s_:32s} train {a_.shape} test {b_.shape}")

pairs (13 features): [('MntGoldProds', 'NumWebPurchases'), ('MntFruits', 'NumDealsPurchases'), ('MntFishProducts', 'NumWebPurchases'), ('NumWebPurchases', 'NumCatalogPurchases')]
pairs (with Teenhome): [('Teenhome', 'NumCatalogPurchases'), ('Teenhome', 'NumWebVisitsMonth'), ('Teenhome', 'MntWines'), ('Teenhome', 'MntMeatProducts'), ('Teenhome', 'NumStorePurchases')]
13 features                      train (1769, 13) test (443, 13)
+ Teenhome (14)                  train (1769, 14) test (443, 14)
13 features + 4 pair nets        train (1769, 13) test (443, 13)
+ Teenhome + 5 Teenhome pairs    train (1769, 14) test (443, 14)


## Model

In [5]:
EPOCHS_MAX, BATCH, LR, WD = 100, 64, 1e-3, 1e-4     # same optimiser settings as V2_0
device = torch.device("cpu")

def run_epochs(model, X, y, loss_fn, epochs, seed, on_epoch=None):
    """Adam + cosine (T_max = 100), mini-batch 64. on_epoch(epoch, model) is called after every epoch."""
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.999), eps=1e-8, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_MAX)
    Xt = torch.from_numpy(np.asarray(X, dtype=np.float32)); yt = torch.from_numpy(np.asarray(y, dtype=np.float32)).view(-1, 1)
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i + BATCH]
            opt.zero_grad(); loss_fn(model(Xt[idx]), yt[idx]).backward(); opt.step()
        sch.step()
        if on_epoch is not None:
            on_epoch(ep, model)
    return model

def smooth(a, w=5):
    return np.convolve(np.pad(a, (w // 2, w // 2), mode="edge"), np.ones(w) / w, mode="valid")

def n_params(m):
    return int(sum(p.numel() for p in m.parameters()))

In [6]:
class SubNet(nn.Module):
    """Linear(n_in -> hid) - ReLU - Linear(hid -> 1). n_in = 1 for a feature, 2 for a feature pair."""
    def __init__(self, n_in=1, hid=64):
        super().__init__()
        self.fc1 = nn.Linear(n_in, hid); self.fc2 = nn.Linear(hid, 1); self.relu = nn.ReLU()
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity="relu"); nn.init.zeros_(self.fc1.bias)
        nn.init.kaiming_normal_(self.fc2.weight, nonlinearity="relu"); nn.init.zeros_(self.fc2.bias)
    def forward(self, x): return self.fc2(self.relu(self.fc1(x)))

class NAM(nn.Module):
    """Sum of per-feature subnets (+ optional 2D subnets for chosen feature pairs, i.e. NA2M) + bias."""
    def __init__(self, n_feat, pair_idx=(), hid=64):
        super().__init__()
        self.pair_idx = [tuple(p) for p in pair_idx]
        self.subs = nn.ModuleList([SubNet(1, hid) for _ in range(n_feat)])
        self.pairs = nn.ModuleList([SubNet(2, hid) for _ in self.pair_idx])
        self.b0 = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        out = sum(sub(x[:, j:j + 1]) for j, sub in enumerate(self.subs)) + self.b0
        for (i, j), net in zip(self.pair_idx, self.pairs):
            out = out + net(x[:, [i, j]])
        return out

DELTA = 0.5
def make_model(v, seed, n_feat=None):
    torch.manual_seed(seed)
    cols, pairs = SETS[v]["cols"], SETS[v]["pairs"]
    idx = [(cols.index(a), cols.index(b)) for a, b in pairs]
    return NAM(n_feat=len(cols), pair_idx=idx).to(device)

In [7]:
INNER_K = 3
def to_z(y_raw_, scaler): return scaler.transform(np.log1p(y_raw_).reshape(-1, 1)).ravel()
def to_dollar(z, scaler): return np.expm1(np.clip(scaler.inverse_transform(np.asarray(z).reshape(-1, 1)).ravel(), 0.0, 15.0))   # clip keeps a bad early epoch finite

def predict_dollar(model, X, scaler):
    model.eval()
    with torch.no_grad():
        return to_dollar(model(torch.from_numpy(np.asarray(X, dtype=np.float32))).numpy().ravel(), scaler)

def fit_eval(variant, Xa, ya, Xb, yb, seed):
    """Epoch count comes from an inner K-fold on (Xa, ya) only; (Xb, yb) is scored once."""
    loss_fn = nn.HuberLoss(delta=DELTA)
    curves = []
    for k, (a, b) in enumerate(KFold(INNER_K, shuffle=True, random_state=seed).split(Xa)):
        sc = StandardScaler().fit(np.log1p(ya[a]).reshape(-1, 1))
        rm = []
        run_epochs(make_model(variant, seed + k), Xa[a], to_z(ya[a], sc), loss_fn, EPOCHS_MAX, seed + k,
                   on_epoch=lambda ep, m, b=b, sc=sc: rm.append(float(np.sqrt(mean_squared_error(ya[b], predict_dollar(m, Xa[b], sc))))))
        curves.append(rm)
    E = int(np.argmin(smooth(np.mean(curves, axis=0)))) + 1   # epoch chosen on inner validation only
    sc = StandardScaler().fit(np.log1p(ya).reshape(-1, 1))
    model = run_epochs(make_model(variant, seed + 99), Xa, to_z(ya, sc), loss_fn, E, seed + 99)
    pred = predict_dollar(model, Xb, sc)
    mse = mean_squared_error(yb, pred)
    row = {"r2": r2_score(yb, pred), "rmse": float(np.sqrt(mse)), "mae": mean_absolute_error(yb, pred), "mse": mse,
           "epochs": E, "n_params": n_params(model)}
    return row, pred

## Sanity Check

In [8]:
a, b = next(KFold(5, shuffle=True, random_state=SEED).split(X_all))
assert len(set(a) & set(b)) == 0, "train / eval rows overlap"
for sname, spec in SETS.items():
    Xa, Xb, ya, yb = prep_split(a, b, sname)
    assert Xa.shape[1] == Xb.shape[1] == len(spec["cols"])
    assert np.allclose(Xa.mean(0), 0, atol=1e-6) and np.allclose(Xa.std(0), 1, atol=1e-6)
    assert np.isfinite(Xa).all() and np.isfinite(Xb).all()
    print(f"{sname:32s} dims {Xa.shape[1]:3d} | pair subnets {len(spec['pairs'])} -> OK")
for sname in SETS:
    m = make_model(sname, 0).eval()
    n = len(SETS[sname]["cols"]); pairs = m.pair_idx
    x = torch.randn(4, n)
    xi = x.clone(); xi[:, 0] += 1.0
    xj = x.clone(); xj[:, 1] += 1.0; xij = xi.clone(); xij[:, 1] += 1.0
    with torch.no_grad():
        cross = (m(xij) - m(xi)) - (m(xj) - m(x))      # zero when features 0 and 1 are only added together
    expect_zero = (0, 1) not in pairs and (1, 0) not in pairs
    assert (cross.abs().max() < 1e-5) == expect_zero, (sname, float(cross.abs().max()), pairs)
    xb = x; yb_ = torch.randn(4, 1); loss = nn.HuberLoss(delta=DELTA)(m.train()(xb), yb_); loss.backward()
    assert torch.isfinite(loss) and all(torch.isfinite(p.grad).all() for p in m.parameters() if p.grad is not None)
    print(f"{sname:32s} parameters {n_params(m):6d} | pair subnets {len(pairs)} | additivity check OK")

13 features                      dims  13 | pair subnets 0 -> OK
+ Teenhome (14)                  dims  14 | pair subnets 0 -> OK
13 features + 4 pair nets        dims  13 | pair subnets 4 -> OK
+ Teenhome + 5 Teenhome pairs    dims  14 | pair subnets 5 -> OK
13 features                      parameters   2510 | pair subnets 0 | additivity check OK
+ Teenhome (14)                  parameters   2703 | pair subnets 0 | additivity check OK


13 features + 4 pair nets        parameters   3538 | pair subnets 4 | additivity check OK
+ Teenhome + 5 Teenhome pairs    parameters   3988 | pair subnets 5 | additivity check OK


## Training

In [9]:
cache = "weights_V5_1/_progress_regression_pytorch.pkl"
state = pickle.load(open(cache, "rb")) if os.path.exists(cache) else {"rows": [], "scores": {}}
done = {(r["model"], r["repeat"], r["fold"]) for r in state["rows"]}
for rep in range(1, REPEATS + 1):
    seed_r = SEED + rep - 1               # same outer folds as V6_1 / V4_1
    for fold, (a, b) in enumerate(KFold(K, shuffle=True, random_state=seed_r).split(X_all), start=1):
        for sname in SETS:
            name = f"NAM (log1p + Huber) | ReLU | {sname}"
            if (name, rep, fold) in done:
                continue
            Xa, Xb, ya, yb = prep_split(a, b, sname)
            t0 = time.time()
            row, s = fit_eval(sname, Xa, ya, Xb, yb, seed_r * 100 + fold)
            extra = {}
            state["rows"].append(dict(model=name, tier="extracurricular", selection="inner validation", repeat=rep, fold=fold,
                                      n_test=int(len(yb)), **{k: round(float(v), 4) for k, v in row.items()},
                                      seconds=round(time.time() - t0, 1), **extra))
            state["scores"].setdefault(f"{name}|r{rep}", np.full(len(y_raw), np.nan))[b] = s
        pickle.dump(state, open(cache, "wb"))
        print(f"repeat {rep} fold {fold} done | rows so far {len(state['rows'])}", flush=True)

repeat 1 fold 1 done | rows so far 4


repeat 1 fold 2 done | rows so far 8


repeat 1 fold 3 done | rows so far 12


repeat 1 fold 4 done | rows so far 16


repeat 1 fold 5 done | rows so far 20


...

repeat 3 fold 1 done | rows so far 44


repeat 3 fold 2 done | rows so far 48


repeat 3 fold 3 done | rows so far 52


repeat 3 fold 4 done | rows so far 56


repeat 3 fold 5 done | rows so far 60


## K-Fold Summary

In [10]:
kdf = pd.DataFrame(state["rows"])
KEYS = ['r2', 'rmse', 'mae']
summary = {m: {k: {"mean": round(float(g[k].mean()), 4), "std": round(float(g[k].std()), 4)} for k in KEYS if k in g and g[k].notna().any()}
           for m, g in kdf.groupby("model")}
print(kdf.groupby("model")["r2"].agg(["mean", "std"]).sort_values("mean", ascending=False).round(4).to_string())
json.dump({"k": K, "repeats": REPEATS, "folds": state["rows"], "summary": summary},
          open("weights_V5_1/regression_pytorch_kfold_metrics.json", "w"), indent=2, default=str)
np.savez("weights_V5_1/regression_pytorch_kfold_scores.npz", **state["scores"])

                                                              mean     std
model                                                                     
NAM (log1p + Huber) | ReLU | + Teenhome + 5 Teenhome pairs  0.7567  0.0743
NAM (log1p + Huber) | ReLU | + Teenhome (14)                0.7477  0.0678
NAM (log1p + Huber) | ReLU | 13 features                    0.7353  0.0651
NAM (log1p + Huber) | ReLU | 13 features + 4 pair nets      0.7246  0.0665
